# Protein–Ligand DMS, BindingDB & BioLiP-175 Quickstart

This notebook explores [`aurlienplissier/protein-ligand-dms`](https://www.kaggle.com/datasets/aurlienplissier/protein-ligand-dms):
1. **137 Protein–Ligand DMS & Enzyme Kinetics Cohorts (`561,059` variants)** (`cohorts.csv` + `dms_tasks/*.parquet`)
2. **Precomputed Zero-Shot Baseline Scores** (`baseline_scores/*_baselines.parquet`) across 10 protein/ligand/structure models
3. **Stage-0 Curated BindingDB Pretraining Corpus** (`bindingdb_pretrain/`: `10,235` proteins, `1.13M` compounds, `1.85M` pairs)
4. **BioLiP-175 `<5 Å` Contact Benchmark & Precomputed LogiCA Dependency Maps** (`biolip_175/`)

Companion code & pretrained `ESM-2 + SELFormer` checkpoints: [AI-SysBio/LogiCA](https://github.com/AI-SysBio/LogiCA/tree/main/applications/protein-ligand) · [Yale-CompBio/logica](https://huggingface.co/Yale-CompBio/logica)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

# Locate Kaggle input directory automatically
candidates = [
    Path("/kaggle/input/protein-ligand-dms"),
    Path("/kaggle/input/datasets/aurlienplissier/protein-ligand-dms"),
]
ROOT = next((p for p in candidates if p.exists()), None)
if ROOT is None:
    import kagglehub
    ROOT = Path(kagglehub.dataset_download("aurlienplissier/protein-ligand-dms"))
print("Dataset root:", ROOT)

In [ ]:
# 1. Inspect the 137-cohort index & split breakdown
cohorts = pd.read_csv(ROOT / "cohorts.csv")
print(f"Total cohorts: {len(cohorts)} (Mutational DMS: {cohorts['is_mutational_dms'].sum()}, WT Kinetics: {(~cohorts['is_mutational_dms']).sum()})")
print(f"Unique UniProt targets: {cohorts['uniprot_id'].nunique()} | LOCO-30% clusters: {cohorts['loco30_cluster'].nunique()}")
display(cohorts.groupby("task_name").agg(cohorts=("cohort_id", "count"), variants=("n_variants", "sum")))

In [ ]:
# 2. Load pure experimental DMS table & join separated baseline scores
task = "ligand_protein_functional"
exp_df = pd.read_parquet(ROOT / f"dms_tasks/{task}.parquet")
base_df = pd.read_parquet(ROOT / f"baseline_scores/{task}_baselines.parquet")
assert len(exp_df) == len(base_df)
merged = pd.concat([exp_df, base_df[[c for c in base_df.columns if c not in exp_df.columns]]], axis=1)

rows = []
for cid, grp in merged.groupby("parent_strain"):
    g = grp.dropna(subset=["dms_score_raw", "esm2_650m_llr", "logica_ligand_llr"])
    if len(g) < 50:
        continue
    rows.append({
        "cohort": cid,
        "n": len(g),
        "ESM2_650M_rho": spearmanr(g["dms_score_raw"], g["esm2_650m_llr"]).statistic,
        "LogiCA_rho": spearmanr(g["dms_score_raw"], g["logica_ligand_llr"]).statistic,
    })
res = pd.DataFrame(rows).sort_values("LogiCA_rho", ascending=False)
display(res.head(10))

In [ ]:
# 3. Audit Stage-0 BindingDB Pretraining Corpus & BioLiP-175 Dependency Maps
bdb_train = pd.read_parquet(ROOT / "bindingdb_pretrain/train_pairs.parquet")
bdb_val = pd.read_parquet(ROOT / "bindingdb_pretrain/val_pairs.parquet")
print(f"BindingDB Stage-1 pairs: train={len(bdb_train):,}, val={len(bdb_val):,}")
print("Train label distribution (1=active <100nM, 0=inactive >10uM, -1=gray zone):")
print(bdb_train["label"].value_counts().to_dict())

dep = pd.read_parquet(ROOT / "biolip_175/dependency_maps/biolip_dependency_profiles.parquet")
auroc_pre = roc_auc_score(dep["contact"], dep["dep_pretrained_aa"])
auroc_ft = roc_auc_score(dep["contact"], dep["dep_finetuned_gt080_aa"])
print(f"BioLiP-175 pooled <5 A contact AUROC (44,337 residues): Pretrained={auroc_pre:.4f}, Protein-Disjoint FT={auroc_ft:.4f}")

fig, ax = plt.subplots(figsize=(7, 3.8))
msum = pd.read_csv(ROOT / "biolip_175/dependency_maps/biolip_model_summary.csv").sort_values("interaction_auroc_mean")
ax.barh(msum["display_name"].str.slice(0, 40), msum["interaction_auroc_mean"], color="#4C78A8")
ax.axvline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("BioLiP-175 Mean Interaction AUROC (<5 A Contacts)")
ax.set_title("Zero-Shot & Fine-Tuned Cross-Modal Binding-Site Localization")
plt.tight_layout()
plt.show()